Simulator: Dataset Offsets
==========================

Multi-wavelength datasets often have offsets between their images, which are due to the different telescope pointings
during the observations.

These offsets are often accounted for during the data reduction process, which aligns the images, however:

 - Certain data reduction pipelines may not perfectly align the images, and the scientist may be unsure what the
 true offset between the images are.

 - Even if the reduction process does align the images, there is still a small uncertainty in the offset due to the
   precision of the telescope pointing which for detailed lens models must be accounted for.

This script simulates a multi-wavelength `Imaging` dataset where the two images have a small offset and a small
relative rotation between them. Both impact the lensing calculation and modeling if not accounted for. The
`modeling` examples show how to include both as free parameters of a `DatasetModel`.

__Contents__

- **Model:** Compose the lens model fitted to the data.
- **Colors:** The colors of the multi-wavelength image, which in this case are green (g-band) and red (r-band).
- **Dataset Paths:** Overview of dataset paths for this example.
- **Simulate:** The pixel-scale of each color image is different meaning we make a list of grids for the simulation.
- **Offset:** Offset the second grid from the first grid by the pixel scale in both the y and x directions.
- **Ray Tracing:** The lens galaxy light at each wavelength has a different intensity, thus we create two lens.
- **Output:** Output each simulated dataset to the dataset path as .fits files, with a tag describing its color.
- **Visualize:** Output a subplot of the simulated dataset, the image and the tracer's quantities to the dataset.
- **Tracer json:** Save the `Tracer` in the dataset folder as a .json file, ensuring the true light profiles, mass.

__Model__

This script simulates multi-wavelength `Imaging` of a 'galaxy-scale' strong lens where:

 - The two datasets have a small offset of half the pixel scale between them.
 - The lens galaxy's light profile is an `Sersic`, which has a different `intensity` at each wavelength.
 - The lens galaxy's total mass distribution is an `Isothermal`; the external shear is an
   `ExternalShear` held in a `MassField`.
 - The source galaxy's light is an `Sersic`, which has a different `intensity` at each wavelength.

Two images are simulated, corresponding to a greener ('g' band) redder image (`r` band).

This is an advanced script and assumes previous knowledge of the core **PyAutoLens** API for simulating images. Thus,
certain parts of code are not documented to ensure the script is concise.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autolens as al
import autolens.plot as aplt

__Colors__

The colors of the multi-wavelength image, which in this case are green (g-band) and red (r-band).

The strings are used for naming the datasets on output.

In [ ]:
waveband_list = ["g", "r"]

__Dataset Paths__

In [ ]:
dataset_type = "multi_dataset"
dataset_label = "imaging"
dataset_name = "dataset_offsets"

dataset_path = Path("dataset") / dataset_type / dataset_label / dataset_name

__Simulate__

The pixel-scale of each color image is different meaning we make a list of grids for the simulation.

In [ ]:
pixel_scales_list = [0.08, 0.12]

grid_list = []

for pixel_scales in pixel_scales_list:
    grid = al.Grid2D.uniform(
        shape_native=(150, 150),
        pixel_scales=pixel_scales,
    )

    over_sample_size = al.util.over_sample.over_sample_size_via_radial_bins_from(
        grid=grid,
        sub_size_list=[32, 8, 2],
        radial_list=[0.3, 0.6],
        centre_list=[(0.0, 0.0)],
    )

    grid = grid.apply_over_sampling(over_sample_size=over_sample_size)

    grid_list.append(grid)


__Offset__

Offset the second grid from the first grid by the pixel scale in both the y and x directions.

In [ ]:
grid_list[1] -= pixel_scales_list[1]

__Rotation__

In addition to the small offset, rotate the second grid by 1.5 degrees counter-clockwise about its
offset point. This emulates a small roll-angle difference between two multi-band exposures (e.g. JWST
NIRCam frames acquired at slightly different telescope orientations). The modeling script recovers this
via the `DatasetModel.grid_rotation_angle` parameter alongside `grid_offset`.

In [ ]:
grid_list[1] = grid_list[1].subtracted_and_rotated_from(offset=(0.0, 0.0), angle=1.5)

Simulate simple Gaussian PSFs for the images in the r and g bands.

In [ ]:
sigma_list = [0.1, 0.2]

psf_list = [
    al.Convolver.from_gaussian(
        convolve_over_sample_size=1,
        shape_native=(11, 11),
        sigma=sigma,
        pixel_scales=grid.pixel_scales,
    )
    for grid, sigma in zip(grid_list, sigma_list)
]

Create separate simulators for the g and r bands.

In [ ]:
background_sky_level_list = [0.1, 0.15]

simulator_list = [
    al.SimulatorImaging(
        exposure_time=300.0,
        psf=psf,
        background_sky_level=background_sky_level,
        add_poisson_noise_to_data=True,
    )
    for psf, background_sky_level in zip(psf_list, background_sky_level_list)
]

__Ray Tracing__

The lens galaxy light at each wavelength has a different intensity, thus we create two lens galaxies for each waveband. 

The lens galaxy's mass (SIE+Shear) is identical for each waveband and included in both lens galaxies in the list..

In [ ]:
intensity_list = [0.05, 1.5]

bulge_list = [
    al.lp.Sersic(
        centre=(0.0, 0.0),
        ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
        intensity=intensity,
        effective_radius=0.8,
        sersic_index=4.0,
    )
    for intensity in intensity_list
]

mass = al.mp.Isothermal(
    centre=(0.0, 0.0),
    einstein_radius=1.6,
    ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
)

lens_galaxy_list = [
    al.Galaxy(
        redshift=0.5,
        bulge=bulge,
        mass=mass,
    )
    for bulge in bulge_list
]

field = al.MassField(
    redshift=0.5,
    shear=al.mp.ExternalShear(gamma_1=0.05, gamma_2=0.05),
)

__Ray Tracing__

The source galaxy at each wavelength has a different intensity, thus we create two source galaxies for each waveband.

In [ ]:
intensity_list = [0.5, 0.7]

source_galaxy_list = [
    al.Galaxy(
        redshift=1.0,
        bulge=al.lp.SersicCore(
            centre=(0.0, 0.0),
            ell_comps=al.convert.ell_comps_from(axis_ratio=0.8, angle=60.0),
            intensity=intensity,
            effective_radius=0.1,
            sersic_index=1.0,
        ),
    )
    for intensity in intensity_list
]

Use these galaxies to setup tracers at each waveband, which will generate each image for the simulated `Imaging` 
dataset.

In [ ]:
tracer_list = [
    al.Tracer(galaxies=[lens_galaxy, source_galaxy], fields=[field])
    for lens_galaxy, source_galaxy in zip(lens_galaxy_list, source_galaxy_list)
]

Lets look at the tracer`s image, this is the image we'll be simulating.

Close inspection of the images shows they are slightly offset by half a pixel from one another.

In [ ]:
for tracer, grid in zip(tracer_list, grid_list):
    aplt.plot_array(array=tracer.image_2d_from(grid=grid), title="Image")

Pass the simulator a tracer, which creates the image which is simulated as an imaging dataset.

In [ ]:
dataset_list = [
    simulator.via_tracer_from(tracer=tracer, grid=grid)
    for grid, simulator, tracer in zip(grid_list, simulator_list, tracer_list)
]

Plot the simulated `Imaging` dataset before outputting it to fits.

In [ ]:
for dataset in dataset_list:
    aplt.subplot_imaging_dataset(dataset=dataset)

__Output__

Output each simulated dataset to the dataset path as .fits files, with a tag describing its color.

In [ ]:
for waveband, dataset in zip(waveband_list, dataset_list):
    aplt.fits_imaging(
        dataset=dataset,
        data_path=Path(dataset_path) / f"{waveband}_data.fits",
        psf_path=Path(dataset_path) / f"{waveband}_psf.fits",
        noise_map_path=Path(dataset_path) / f"{waveband}_noise_map.fits",
        overwrite=True,
    )

__Visualize__

Output a subplot of the simulated dataset, the image and the tracer's quantities to the dataset path as .png files.

In [ ]:
for waveband, dataset in zip(waveband_list, dataset_list):
    aplt.subplot_imaging_dataset(dataset=dataset)
    aplt.plot_array(array=dataset.data, title="Data")

for waveband, grid, tracer in zip(waveband_list, grid_list, tracer_list):
    aplt.subplot_tracer(tracer=tracer, grid=grid)
    aplt.subplot_galaxies_images(tracer=tracer, grid=grid)

    aplt.plot_array(array=tracer.image_2d_from(grid=grid), title="Image")

__Tracer json__

Save the `Tracer` in the dataset folder as a .json file, ensuring the true light profiles, mass profiles and galaxies
are safely stored and available to check how the dataset was simulated in the future. 

This can be loaded via the method `tracer = al.from_json()`.

In [ ]:
[
    al.output_to_json(obj=tracer, file_path=Path(dataset_path, f"{color}_tracer.json"))
    for color, tracer in zip(waveband_list, tracer_list)
]

The dataset can be viewed in the folder `autolens_workspace/dataset/multi_dataset/imaging/dataset_offsets`.